# HyperKT Objective 01 — Stage 7: Reproducibility & Leakage Audit
**Runtime: CPU (Python 3), no accelerator.** Inputs: Stage 2 ZIP and Stage 6 ZIP. This audit does **not** score the held-out test set. It checks saved arrays, checkpoint structure, validation metrics and split integrity. Passing this audit does not prove that the model's causal attention implementation is correct; that requires source-level tests in a later stage.

In [ ]:
import numpy as np, pandas as pd, torch, json, zipfile, io, hashlib, os, shutil, pathlib
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
print('PyTorch',torch.__version__,'CUDA available:',torch.cuda.is_available())
try:
 from google.colab import files
 uploaded=files.upload()
except ImportError:
 print('Running outside Colab: place ZIP files in working directory.')


In [ ]:
from pathlib import Path
files_here=list(Path('.').glob('*.zip')) + list(Path('/content').glob('*.zip'))
def select_zip(stage):
 matches=list(dict.fromkeys(p for p in files_here if f'hyperkt_stage{stage}' in p.name.lower()))
 assert matches, f'Missing Stage {stage} ZIP; upload it and rerun this cell.'
 return matches[-1]
z2=select_zip('2'); z6=select_zip('6')
print('Stage 2:',z2,'Stage 6:',z6)
def load_json(z,name):
 with zipfile.ZipFile(z) as f: return json.loads(f.read(name))
def load_npz(z,name):
 with zipfile.ZipFile(z) as f:
  with np.load(io.BytesIO(f.read(name)),allow_pickle=False) as d:return {k:d[k].copy() for k in d.files}
report2=load_json(z2,'stage2_report.json'); manifest=load_json(z2,'manifest.json'); report6=load_json(z6,'report.json')
seq={s:load_npz(z2,f'{s}_sequences.npz') for s in ['train','val','test']}
pred=load_npz(z6,'validation_predictions.npz')
print('Loaded sequence splits and saved validation predictions')


In [ ]:
checks={}; details={}
def check(name,condition,detail=''):
 checks[name]=bool(condition);details[name]=str(detail)
 print(('PASS' if condition else 'FAIL'),name,detail)
required={'history_skill','history_correct','target_skill','target_correct','mask','user_id'}
for s,d in seq.items():
 check(f'{s}_required_keys',required.issubset(d),str(d.keys()))
 shapes={k:d[k].shape for k in ['history_skill','history_correct','target_skill','target_correct','mask']}
 check(f'{s}_aligned_shapes',len(set(shapes.values()))==1,shapes)
 check(f'{s}_max_length',d['mask'].shape[1]==report2['max_len'])
 check(f'{s}_target_count',int(d['mask'].sum())==report2['splits'][s]['targets'],int(d['mask'].sum()))
 check(f'{s}_binary_targets',np.isin(d['target_correct'][d['mask']],[0,1]).all())
 check(f'{s}_binary_history',np.isin(d['history_correct'][d['mask']],[0,1]).all())
 check(f'{s}_skill_ids_valid',np.all(d['target_skill'][d['mask']]>=2) and np.all(d['target_skill'][d['mask']]<=150))
 check(f'{s}_nonempty_targets',np.all(d['mask'].sum(axis=1)>0))
 check(f'{s}_mask_contiguous',np.all(np.diff(d['mask'].astype(np.int8),axis=1)<=0))


In [ ]:
users={s:set(map(str,d['user_id'])) for s,d in seq.items()}
for a,b in [('train','val'),('train','test'),('val','test')]:
 check(f'disjoint_{a}_{b}',not (users[a]&users[b]),f'overlap={len(users[a]&users[b])}')
for s in seq:
 check(f'{s}_unique_users_within_windows',len(users[s])<=len(seq[s]['user_id']),f'{len(users[s])} distinct users / {len(seq[s]["user_id"])} windows')
 check(f'{s}_manifest_user_subset',users[s].issubset(set(map(str,manifest['splits'][s]))),f'{len(users[s])} sequence users')
 check(f'{s}_user_count_not_exceed_manifest',len(users[s])<=report2['splits'][s]['students'])
# Verify that within each window, next-target at position t equals next history at position t+1.
# This tests local shifting only; cross-window chronology needs source-level auditing.
for s,d in seq.items():
 valid=d['mask'][:,:-1] & d['mask'][:,1:]
 skill_ok=np.all((d['target_skill'][:,:-1]==d['history_skill'][:,1:])[valid])
 correct_ok=np.all((d['target_correct'][:,:-1]==d['history_correct'][:,1:])[valid])
 check(f'{s}_local_next_step_shift',skill_ok and correct_ok,f'checked {int(valid.sum())} adjacent positions')


In [ ]:
def ece(y,p,bins=15):
 edges=np.linspace(0,1,bins+1);idx=np.minimum(np.digitize(p,edges[1:-1]),bins-1)
 return float(sum((idx==i).mean()*abs(y[idx==i].mean()-p[idx==i].mean()) for i in range(bins) if np.any(idx==i)))
y=pred['y'];check('validation_prediction_count',len(y)==int(seq['val']['mask'].sum()))
check('validation_label_binary',np.isin(y,[0,1]).all())
# Compare saved labels with the flattened validation mask order, without accessing test labels.
y_expected=seq['val']['target_correct'][seq['val']['mask']]
check('validation_prediction_label_alignment',np.array_equal(y,y_expected),'Requires original prediction flattening order')
metric_recalc={}
for key,reported in [('p_deterministic','selected_deterministic_validation'),('p_mc30','selected_mc30_validation')]:
 p=pred[key];check(f'{key}_valid_probabilities',np.isfinite(p).all() and np.all((p>=0)&(p<=1)))
 m={'auc':float(roc_auc_score(y,p)),'nll':float(log_loss(y,p,labels=[0,1])),'brier':float(brier_score_loss(y,p)),'ece_15':ece(y,p),'n_targets':int(len(y))}
 metric_recalc[key]=m
 for k,v in m.items():
  check(f'{key}_{k}_matches_report',abs(v-report6[reported][k])<1e-6,f'calculated={v:.9f}, saved={report6[reported][k]:.9f}')
print(json.dumps(metric_recalc,indent=2))


In [ ]:
with zipfile.ZipFile(z6) as f:
 ckpt=torch.load(io.BytesIO(f.read('stage6_model.pt')),map_location='cpu',weights_only=False)
state=ckpt.get('state_dict',{})
check('checkpoint_has_state_dict',bool(state))
check('checkpoint_all_tensors_finite',all(torch.isfinite(v).all().item() for v in state.values() if torch.is_tensor(v)))
check('checkpoint_selected_epoch',ckpt.get('selected_epoch')==report6['selected_epoch'])
check('test_unused_reported',report6.get('test_used') is False,'Protocol assertion, not independent proof')
print('State keys:',len(state),'first 15:',list(state)[:15])
print('WARNING: A checkpoint cannot establish causal masking correctness without the original model forward code.')


In [ ]:
from datetime import datetime,timezone
out=Path('/content/hyperkt_stage7_audit' if Path('/content').exists() else 'hyperkt_stage7_audit');out.mkdir(parents=True,exist_ok=True)
summary={'stage':7,'audit_time_utc':datetime.now(timezone.utc).isoformat(),'input_files':{},'checks':checks,'details':details,'metrics_recomputed':metric_recalc,'pass_count':sum(checks.values()),'fail_count':sum(not x for x in checks.values()),'limitations':['No test labels or test predictions evaluated','Causal attention implementation not verified without model source','Cross-window chronological target uniqueness not fully verified','Reported test_used=false is a protocol assertion']}
for z in (z2,z6):summary['input_files'][z.name]=hashlib.sha256(z.read_bytes()).hexdigest()
(out/'audit_report.json').write_text(json.dumps(summary,indent=2))
pd.DataFrame([{'check':k,'passed':v,'detail':details[k]} for k,v in checks.items()]).to_csv(out/'audit_checks.csv',index=False)
(out/'README.txt').write_text('HyperKT Stage 7 static reproducibility audit. Test predictions not computed. Review failed checks and limitations before final test.')
archive=shutil.make_archive(str(out),'zip',out)
print('PASS',summary['pass_count'],'FAIL',summary['fail_count']);print('Output:',archive)
try:
 from google.colab import files
 files.download(archive)
except ImportError: pass
